In [ ]:
!pip install pandapower rich

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 22.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 3.4 MB/s eta 0:00:00
  Created wheel for pandapower: filename=pandapower-2.14.11-py3-none-any.whl size=13131028 sha256=c52a37e0089770f6723ee2cb4a2eb5abeb3a7a87cea5f83a62117e2d811a175c
  Stored in directory: /root/.cache/pip/wheels/e6/7d/bf/a0b85b42ca2601f8b07aa66ce038d2c92a74eb9c678551aebd
Successfully built pandapower




---
# ATENÇÂO ao Rainer do futuro (30/01/2025)
---

1) Sei muito bem dos prints do codigo, tem muitos comentarios que podem ser retirados

mas para usar o sistema de logging (feito por mim) é so usar a função **Debug = True** e digitar

```py
rede.console.log("mensagem")
```

2) Verificar se o algoritmo que gera a matriz de cenários está expandindo a janela da forma correta para horários que passam da meia noite.

3) Estou com um codigo em streamlit para frontend nativa em python para colocar os dados de entrada e rodar a automação

4) Dashboard em streamlit esta funcionando tanto para essa automação do pandapower quanto no antigo algoritimo evolutivo. Na minha cabeça eu passo um excel de agendamentos e o sistema é capaz de calcular as violaçoes e cenarios e me dar o melhor cenario. Tambem pensei em fazer um sketch em protoboard Iot com liga e desliga leds simulando cada ramos da rede eletrica (apenas diversão para ferias e treinando eletronica com conectividade a internet)

## Classe

---



In [ ]:
import numpy as np
import pandapower as pp
import pandapower.networks as pw
import pandas as pd

from rich.console import Console
from rich.theme import Theme
from rich.traceback import install

install()

class Logger:
    def __init__(self):
        self.console = Console(theme=Theme({
            "success": "bold green",
            "warning": "yellow",
            "error": "bold red",
            "info": "white"  # Added "info" level for default blue color
        }))

    def log(self, message, level="info"):  # Changed default level to "info"
        """Logs a message with the specified level and color."""
        if level == "success":
            self.console.print(f"[success]{message}[/]")
        elif level == "warning":
            self.console.print(f"[warning]{message}[/]")
        elif level == "error":
            self.console.print(f"[error]{message}[/]")
        else:
            self.console.print(f"[info]{message}[/]") # Changed to "info" to use blue color



class RedeEletricaPandaPower:
    def __init__(self, network_name, debug=False):
        self.net = self.carregar_redes_padrao(network_name)
        self.debug = debug
        self.console = Logger()

        #metoodos
        self.criar_mapeamento_ramos()

        # global
        self.pesos = {
            "tensao": {"min": 100, "max": 100},
            "loading_linhas": 100,
            "loading_trafos": 150,
            "demanda": 99,
        }
        self.agendamento = pd.DataFrame()
        self.contingencia= pd.DataFrame()

    def carregar_redes_padrao(self,network_name = "14"):
        #!todo -> Switch para as redes disponiveis na lib
        match network_name:
            case "14":
                network = pw.case14()
            case "4gs":
                #This is the 4 bus example
                network = pw.case4gs()
            case "5":
                #This is the 5 bus example
                network = pw.case5()
            case "6ww":
                # It represents the 6 bus example from pp. 104, 112, 119, 123-124, 549
                network = pw.case6ww()
            case "9":
                network = pw.case9()
            case "24":
                #The IEEE 24-bus reliability test system was developed by the IEEE reliability subcommittee and published in 1979.
                network = pw.case24_ieee_rts()
            case "30":
                network = pw.case30()
            case "300":
                network = pw.case300()
            case "case30":
                #This function calls the json file case_ieee30.json which data origin is MATPOWER.
                network = pw.case_ieee30()
            case "33":
                # Calls the json file case33bw.json which data is provided by MATPOWER.
                network = pw.case33bw()
            case "39":
                # Calls the json file case39.json which data origin is PYPOWER.
                network = pw.case39()
            case "57":
                # This function provides the ieee case57 network with the data origin PYPOWER
                network = pw.case57()
            case _:
                print("Rede não encontrada, forneça o numero como string")
                network = None

        return network

    def criar_mapeamento_ramos(self):
        """Mapeia pares de barramentos para índices de linhas e trafos"""
        self.mapeamento_ramos = {
            'linhas': {},
            'trafos': {}
        }

        # Linhas
        for idx, row in self.net.line.iterrows():
            key = tuple(sorted((row['from_bus'], row['to_bus'])))
            self.mapeamento_ramos['linhas'][key] = idx

        # Transformadores
        for idx, row in self.net.trafo.iterrows():
            key = tuple(sorted((row['hv_bus'], row['lv_bus'])))
            self.mapeamento_ramos['trafos'][key] = idx

        return self.mapeamento_ramos



    def validar_dados(self, df_agendamento, df_contingencia):
        """Valida consistência dos dados antes de processar"""
        # Verifica colunas obrigatórias
        required_agendamento = ["ramo", "inicio", "duracao", "prioridade"]
        if not all(col in df_agendamento.columns for col in required_agendamento):
            raise ValueError("Colunas faltantes no agendamento_df")

        # Verifica existência dos ramos
        for _, row in df_agendamento.iterrows():
            ramo = tuple(sorted(row['ramo']))
            if not (ramo in self.mapeamento_ramos['linhas'] or ramo in self.mapeamento_ramos['trafos']):
                raise ValueError(f"Ramo {row['ramo']} não existe na rede")

        self.agendamento = df_agendamento
        self.contingencia = df_contingencia




    def gerar_cenarios_completos(self, matriz_cenarios, contingencia_df):
        """Combina cenários de agendamento com contingências"""
        cenarios_completos = []

        for cenario in matriz_cenarios:
            for _, cont in contingencia_df.iterrows():
                # Cria cópia segura do cenário original
                novo_cenario = {
                    'perfil': cenario[0],
                    'desligamentos': cenario[1:],
                    'contingencia': (cont['from'], cont['to'])
                }
                cenarios_completos.append(novo_cenario)

        return cenarios_completos



    def avaliar_cenario_v3(self, cenario, agendamento_df):
        """Avalia um cenário completo com contingência"""
        try:
            # Resetar rede
            self.religar_todos_os_ramos_agendamento()

            # Aplicar desligamentos programados
            self.desligar_elementos_agendamento(cenario['desligamentos'], agendamento_df)

            # Aplicar contingência
            ramo_cont = tuple(sorted(cenario['contingencia']))
            self.desligar_elementos_agendamento([1], pd.DataFrame([{'ramo': ramo_cont}]))  # Força desligamento

            # Ajustar carga e executar fluxo
            self.ajustar_cargas(cenario['perfil'])
            if not self.executar_fluxo_de_carga():
                return float('inf')  # Penalidade máxima se não convergir

            return self.calcular_violacoes_fitness()[0]

        except Exception as e:
            self.log(f"Erro no cenário {cenario}: {e}", 'error')
            return float('inf')


    def hashtableindex (self, carregamento, n_carregamentos, contingencia, n_contingencias, desligamentos):
        """"
        Recebe os dados do cenário e retorna o índice da tabela hash correspondente
        carregamento -> inteiro de 1 a numero de carregamentos
        n_carregamentos -> inteiro com o número total de carregamentos
        contingencia -> inteiro de 1 a numero de contingencias
        n_contingencias -> total de contingencias
        desligamentos -> vetor linha com ndeslig elementos booleanos
        """
        num_desligamentos= len(desligamentos)

        #converte o vetor binário em inteiro de forma eficiente
        #https://stackoverflow.com/questions/24560596/fastest-way-to-convert-a-binary-listor-array-into-an-integer-in-python
        digits = ['0', '1']


        k = int("".join([ digits[y] for y in desligamentos ]), 2)

        return (k * (n_carregamentos) * (n_contingencias) ) + ((carregamento-1) * (n_contingencias))  + (contingencia-1)






    #==============================================================================================================================================================

    #! UTILS
    def log(self, mensagem,level="info"):
        if self.debug:
            self.console.log(mensagem,level)


    def show_status(self):


        print("\nStatus Linhas")
        display(self.net.line[["from_bus","to_bus","in_service"]])

        print("\nStatus Transformadores")
        display(self.net.trafo[["hv_bus","lv_bus","in_service"]])

        if self.debug:
            print("="*80)
            print("Rede atual")
            print("="*80)

            ## Barramentos
            print("\nTensões nos Barramentos (pu):")
            display(self.net.res_bus[['vm_pu']])

            ## linhas
            print("\nPorcentagem de Carga nas Linhas (%):")
            display(self.net.res_line[['loading_percent']])

            print("\nPotência Aparente nas Linhas (MVA):")
            display(self.net.res_line[['p_from_mw', 'q_from_mvar']])


            # transformadores
            print("\nPotencia aparente nos transformadores")
            display(self.net.res_trafo[['p_hv_mw', 'q_hv_mvar', 's_aparente_hv_mva', 'p_lv_mw', 'q_lv_mvar', 's_aparente_lv_mva']])


            print("\nPorcentagem de Carga nos transformadores (%):")
            display(self.net.res_trafo[['loading_percent']])

            print("="*80)



    def extract_dataset(self):
        """Extrai os dados da rede em um DataFrame"""
        try:
            dataframe = pd.DataFrame()

            # Dados dos barramentos
            dataframe["tensao_nos_barramentos"] = self.net.res_bus[['vm_pu']]

            # Dados das linhas
            dataframe["potencia_aparente_nas_linhas"] = (self.net.res_line['p_from_mw']**2 +
                                                        self.net.res_line['q_from_mvar']**2)**0.5
            # loading
            dataframe["porcentagem_de_carga_nas_linhas"] = self.net.res_line[['loading_percent']]

            # Dados dos transformadores
            dataframe["potencia_aparente_nos_transformadores"] = self.net.res_trafo[['p_hv_mw']]
            dataframe["porcentagem_de_carga_nos_transformadores"] = self.net.res_trafo[['loading_percent']]

            return dataframe

        except Exception as e:
            print(f"Erro ao extrair dataset: {e}")

            return pd.DataFrame()  # Retorna DataFrame vazio em caso de erro

    #! Otimiazação
    def calcular_loading_linhas(self):
        """Calcula violações nos elementos da rede elétrica.
        """
        violacoes = []

        for idx, loading in enumerate(self.net.res_line.loading_percent):
            self.log(f"Linha {idx}: {loading:.2f}% carregada.")

            if loading > 100:
                violacoes.append(f"Linha {idx} sobrecarregada: {loading:.2f}%")

        print(f"\n\nViolações de loading nas linhas calculadas: {violacoes}")
        return violacoes

    def calcular_violacoes_fitness(self):
        """
        Calcula as violações nos barramentos, linhas e transformadores.

        Considerando um peso para cada grandeza : dois pesos para tensão (max e min) e outro para loading_percent das linhas.

        Somar (valor - limite max ou limite min - valor) para calcular a aptidão daquele cenário.

        Retornar o somatório de todas as violações, mas ao soma cada violação você deve multiplicar por um peso para determinar a aptidão do cenário.

        """
        violacoes = {
            "tensao_barramentos_min": 0,
            "tensao_barramentos_max": 0,
            "loading_linhas": 0,
            "loading_trafos": 0,
        }

        # Verificar tensões nos barramentos (pu) - check
        for idx, row in self.net.res_bus.iterrows():
            # Certifique-se de que a tensão está em pu
            tensao_pu = row["vm_pu"]

            limite_max = self.net.bus.at[idx, "max_vm_pu"]
            limite_min = self.net.bus.at[idx, "min_vm_pu"]

            if tensao_pu > limite_max:
                violacoes["tensao_barramentos_max"] += tensao_pu - limite_max
            elif tensao_pu < limite_min:
                violacoes["tensao_barramentos_min"] += limite_min - tensao_pu

        # Verificar carregamento das linhas
        for idx, row in self.net.res_line.iterrows():

            carregamento = row["loading_percent"] * 100
            limite_max = self.net.line.at[idx, "max_loading_percent"]

            if carregamento > limite_max:
                self.log("\n\nUltrapassou limite maximo nas linhas",level = "warning")
                self.log(f"{carregamento:.2f} > {limite_max} %",level = "warning")
                #RZ - as violações também devem considerar 100% = 1, deve-se dividir
                violacoes["loading_linhas"] += (carregamento - limite_max) / 100

        # Verificar carregamento dos transformadores
        for idx, row in self.net.res_trafo.iterrows():
            carregamento = row["loading_percent"] * 100
            limite_max = self.net.trafo.at[idx, "max_loading_percent"]

            if carregamento > limite_max:
                self.log("\n\nUltrapassou limite maximo nos transformadores",level = "warning")
                self.log(f"{carregamento:.2f} > {limite_max} %",level = "warning")
                #RZ - as violações também devem considerar 100% = 1, deve-se dividir
                violacoes["loading_trafos"] += (carregamento - limite_max) / 100

        #! TODO -> PASSAR OS PESOS NA ISNTANCIA DO OBJETO COM VALOR DEFAULT
        """
        Pesos das violações : (Pdem=99,Pv = 100, Pn = 100 e Pe = 150)
        onde PV é a violação de tensão max e min,
        Pdem é para não convergência do fluxo
        Pn e Pe são para o fluxo de potência (pode usar só Pn que depois explico o que é Pe).
        """


        fitness = (
            self.pesos["tensao"]["min"] * violacoes["tensao_barramentos_min"]
            + self.pesos["tensao"]["max"] * violacoes["tensao_barramentos_max"]
            + self.pesos["loading_linhas"] * violacoes["loading_linhas"]
            + self.pesos["loading_trafos"] * violacoes["loading_trafos"]
        )

        #pega as violacoes e transforma em um DF
        violacoes_df = pd.DataFrame([violacoes])

        if self.debug:
            print("\nTotal de violações e salvando num banco de dados...")
            display(violacoes_df)
            self.console.log(f"\n\nAptidão do cenário nos barramentos, linhas e transformadores ", level = "success")
            self.console.log(f"VIOLAÇÃO TOTAL  = {fitness:.2f}\n", level = "success")

        return fitness, violacoes_df


    def calcular_perfil(self,j, ls, le, ms, me, hs, he):
        """
        Calcula o perfil de carregamento (leve, médio ou pesado) para a hora `j`.

        Args:
            j (int): Hora atual.
            ls, le, ms, me, hs, he (int): Limites de horários para os perfis de carga.

        Returns:
            int: Perfil de carregamento (1 = leve, 2 = padrão, 3 = pesado).
            perfil de carga media é igual IEEE_14
            perfil de carga pesada = multiplicar todas as potencias ativas e reativas, identificando os elementos das estruturas de self.net da classe RedeEletrica do pandapower

        """
        if ls <= j % 24 < le:
            return 1  # Leve
        elif ms <= j % 24 < me:
            return 2  # Médio
        elif hs <= j % 24 < he:
            return 3  # Pesado
        return 0  # Fora dos horários definidos


    def avalia_cenarios_matlab_rainer(self, m, x, duracao, ls, le, ms, me, hs, he):
        """
        Avalia cenários de agendamento com base em desligamentos.

        Args:
            m (int): Horas de duração da janela de tempo.
            x (list): Vetor de horários iniciais dos desligamentos (valores de 0 a m-1).
            duracao (list): Vetor de duração em horas de cada desligamento.
            ls, le, ms, me, hs, he (int): Limites iniciais e finais dos horários de carregamento leve, médio e pesado.

        Returns:
            list: Matriz que armazena todos os cenários do agendamento.
        """

        #! TODO HERE
        # m = max(inicio + duracao for inicio, duracao, _ in desligamentos)

        n = len(x)  # Número de desligamentos
        Scen = []  # Inicializa matriz de saída

        # Ajusta limite da janela de tempo se algum desligamento terminar fora da janela
        for i in range(n):
            if m < (x[i] + duracao[i]):
                m = x[i] + duracao[i]

        # Inicializa matrizes auxiliares
        S = np.zeros((n, m), dtype=int)
        Top = np.zeros(m, dtype=int)

        # =================== Avaliando desligamentos por hora =================
        for j in range(m):  # Para cada hora
            for i in range(n):  # Para cada desligamento
                if x[i] <= j < (x[i] + duracao[i]):
                    S[i, j] = 1
                Top[j] += S[i, j] * (2 ** i)

        # =================== Avaliando cenários =================
        numcenarios = 0
        for j in range(m):
            if j == 0:  # Condição inicial
                if Top[j] > 0:  # Se há pelo menos um desligamento ativo
                    numcenarios += 1
                    perfil = self.calcular_perfil(j, ls, le, ms, me, hs, he)
                    Scen.append([perfil] + S[:, j].tolist())
            else:
                if Top[j] != Top[j - 1] and Top[j] > 0:  # Nova topologia
                    numcenarios += 1
                    perfil = self.calcular_perfil(j, ls, le, ms, me, hs, he)
                    Scen.append([perfil] + S[:, j].tolist())
                else:  # Mesmo cenário, mas perfil pode mudar
                    if j - 1 in [ms, hs]:
                        perfil = self.calcular_perfil(j, ls, le, ms, me, hs, he)
                        Scen[-1][0] = max(Scen[-1][0], perfil)

        return Scen

    def avalia_cenarios(self, horas: int, hora_inicio: list, duracao: list, ls, le, ms, me, hs, he , debug = False):
        """
        Args:
            horas (int): Horas de duração da janela de tempo.
            hora_inicio (list): Vetor de horários iniciais dos desligamentos (valores de 0 a m-1).
            duracao (list): Vetor de duração em horas de cada desligamento.
            ls, le, ms, me, hs, he (int): Limites iniciais e finais dos horários de carregamento leve, médio e pesado.

        Returns:
            list: Matriz que armazena todos os cenários do agendamento.
        """

        matriz_cenarios = []
        num_desligamentos = len(hora_inicio)

        # Ajusta limite da janela de tempo se algum desligamento terminar fora da janela
        for i in range(num_desligamentos):
            if horas < (hora_inicio[i] + duracao[i]):
                horas = hora_inicio[i] + duracao[i]

        # Inicializa matrizes auxiliares
        matriz_desligamentos_horas = np.zeros((num_desligamentos, horas), dtype=int)
        matriz_horas = np.zeros(horas, dtype=int)

        # =================== Avaliando desligamentos por hora =================
        for j in range(horas):  # Para cada hora
            for k in range(num_desligamentos):  # Para cada desligamento
                if hora_inicio[k] <= j < (hora_inicio[k] + duracao[k]):
                    matriz_desligamentos_horas[k, j] = 1
                matriz_horas[j] += matriz_desligamentos_horas[k, j] * (2 ** k)

        # =================== Avaliando cenários =================
        for horario in range(horas):
            if horario == 0:  # Condição inicial
                if matriz_horas[horario] > 0:
                    # Se há pelo menos um desligamento ativo
                    perfil = self.calcular_perfil(horario, ls, le, ms, me, hs, he)
                    matriz_cenarios.append([perfil] + matriz_desligamentos_horas[:, horario].tolist())
            else:
                if matriz_horas[horario] != matriz_horas[horario - 1] and matriz_horas[horario] > 0:
                    # Nova topologia
                    perfil = self.calcular_perfil(horario, ls, le, ms, me, hs, he)
                    matriz_cenarios.append([perfil] + matriz_desligamentos_horas[:, horario].tolist())
                else:
                    # Mesmo cenário, mas perfil pode mudar
                    if horario - 1 in [ms, hs] and matriz_cenarios:  # Verifica se matriz_cenarios não está vazia
                        perfil = self.calcular_perfil(horario, ls, le, ms, me, hs, he)
                        matriz_cenarios[-1][0] = max(matriz_cenarios[-1][0], perfil)

        if self.debug:
            self.log("\nMatriz Cenarios:")
            for linha in matriz_cenarios:
                self.log(linha)
            self.log(f"Avaliando um total de {len(matriz_cenarios)} cenários ")


        return matriz_cenarios

    #! Pandapower New metodos
    def executar_fluxo_de_carga(self):
        """
        Executa o fluxo de carga na rede elétrica usando o algoritmo Newton-Raphson.

        Retorna:
            bool: True se o fluxo de carga convergiu, False caso contrário.
        """
        try:
            pp.runpp(self.net, algorithm="nr")
            self.log("\nFluxo de potência executado com sucesso.",level = "success")

            return True
        except pp.LoadflowNotConverged:
            self.console.log("\nErro: Fluxo de potência não convergiu.", level = "error")
            Pdem = 99

            self.calcular_violacoes_fitness()


            return False

    def ajustar_cargas(self, perfil):
        """Ajusta as cargas conforme o perfil (1 = leve, 2 = médio, 3 = pesado)."""
        tipo = ""

        if perfil == 1:
            fator = 0.941  # Carga leve

            tipo = "leve"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        elif perfil == 2:
            fator = 1.0  # Carga média (IEEE14)

            tipo = "media"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        elif perfil == 3:
            fator = 1.177  # Carga pesada

            tipo = "pesada"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        else:
            fator = 1.0  # Perfil padrão (IEEE14)

            tipo = "padrão"
            self.log(f"Ajustando cargas para o perfil {perfil} ({tipo})...")

        # usando o scaling
        self.net.load["p_mw"] *= fator
        self.net.load["q_mvar"] *= fator

        self.log("Cargas ajustadas.", level = "success")

    def desligar_elementos_agendamento(self, estados):
        """Desliga os elementos (linhas e trafos) com base no cenário."""

        linhas_desligar = []
        trafos_desligar = []

        for i, estado in enumerate(estados):
            if estado == 1:  # Verifica se o ramo deve ser desligado



                ramo = self.agendamento.iloc[i]["ramo"]

                #ramo = agendamento_df.iloc[i]["ramo"]  # Obtém o ramo da tabela

                #self.log("Ramo selecionado",ramo)

                for k in range(len(self.net.line)):
                    # TODO Verifica se o ramo é uma linha ou um transformador
                    if (self.net.line["from_bus"][k] == ramo[0] and self.net.line["to_bus"][k] == ramo[1]) or (self.net.line["from_bus"][k] == ramo[1] and self.net.line["to_bus"][k] == ramo[0] ):
                        linhas_desligar.append(ramo)  # Adiciona o ramo à lista de linhas

                for k in range(len(self.net.trafo)):
                    if (self.net.trafo["hv_bus"][k] == ramo[0] and self.net.trafo["lv_bus"][k] == ramo[1]) or (self.net.trafo["hv_bus"][k] == ramo[1] and self.net.trafo["lv_bus"][k] == ramo[0] ):
                        trafos_desligar.append(ramo)  # Adiciona o ramo à lista de trafos

        self.log(f"\n\nLinhas a serem desligadas: {linhas_desligar}")
        self.log(f"Trafos a serem desligados: {trafos_desligar}\n")

        # Desliga as linhas e trafos encontrados
        self.desligar_elementos(linhas_desligar, trafos_desligar)



    def desligar_contingencia(self, ramo):
        linhas_desligar = []
        trafos_desligar = []


        self.log("Ramo selecionado",ramo)

        for k in range(len(self.net.line)):
            # TODO Verifica se o ramo é uma linha ou um transformador
            if (self.net.line["from_bus"][k] == ramo[0] and self.net.line["to_bus"][k] == ramo[1]) or (self.net.line["from_bus"][k] == ramo[1] and self.net.line["to_bus"][k] == ramo[0] ):
                    linhas_desligar.append(ramo)  # Adiciona o ramo à lista de linhas

        for k in range(len(self.net.trafo)):
            if (self.net.trafo["hv_bus"][k] == ramo[0] and self.net.trafo["lv_bus"][k] == ramo[1]) or (self.net.trafo["hv_bus"][k] == ramo[1] and self.net.trafo["lv_bus"][k] == ramo[0] ):
                    trafos_desligar.append(ramo)  # Adiciona o ramo à lista de trafos

        self.log(f"\n\nLinhas a serem desligadas: {linhas_desligar}")
        self.log(f"Trafos a serem desligados: {trafos_desligar}\n")

        # Desliga as linhas e trafos encontrados
        self.desligar_elementos(linhas_desligar, trafos_desligar)

    def desligar_elementos(self, linhas_desligar, trafos_desligar):
        # Itera pelas linhas a serem desligadas e as desliga na rede
        if not linhas_desligar:
            self.log("Nenhuma linha para desligar")

        else:
            for l in linhas_desligar:

                # Encontra o índice da linha com base em from_bus e to_bus
                index_linha = self.net.line.loc[(self.net.line['from_bus'] == l[0]) & (self.net.line['to_bus'] == l[1])].index

                # Verifica se o índice foi encontrado (CORRIGIDO AQUI PVRV)
                if not index_linha.empty:
                    # Desliga a linha usando o índice encontrado
                    self.net.line.loc[index_linha, 'in_service'] = False

                    #print(f"Linha {l} desligada com sucesso.")


                else:
                    print(f"Linha {l} não encontrada na rede.")


        # Itera pelos transformadores a serem desligados e os desliga na rede
        if not trafos_desligar:
            self.log("Nenhum transformador para desligar")
        else:
            for t in trafos_desligar:

                # Encontra o índice da linha com base em from_bus e to_bus
                index_trafo = self.net.trafo.loc[(self.net.trafo['hv_bus'] == t[0]) & (self.net.trafo['lv_bus'] == t[1])].index

                # Verifica se o índice foi encontrado
                if not index_trafo.empty:
                    # Desliga a linha usando o índice encontrado
                    self.net.trafo.loc[index_trafo, 'in_service'] = False

                    #print(f"Transformador {t} desligado com sucesso.")

                else:
                    print(f"Transformador {t} não encontrado na rede.")


    #! Old Pandapower
    def desligar_varias_linhas(self, linhas, show_prints = False):
        """Desliga várias linhas passadas como lista e executa o fluxo de carga para imprimir resultados."""
        for linha_idx in linhas:
            if linha_idx in self.net.line.index:
                self.net.line.at[linha_idx, 'in_service'] = False
                print(f"Linha {linha_idx} desligada.")
            else:
                print(f"Linha {linha_idx} não encontrada na rede.")

        # Executa o fluxo de carga e imprime resultados para as linhas desligadas
        if show_prints:
            self.executar_fluxo_de_carga()
            self.imprimir_resultados()

    def desligar_transformadores(self, transformadores, show_prints=False):
        """Desliga vários transformadores e executa o fluxo de carga."""
        for transformador in transformadores:
            if transformador in self.net.trafo.index:
                self.net.trafo.at[transformador, 'in_service'] = False
                print(f"Transformador {transformador} desligado.")
            else:
                print(f"Transformador {transformador} não encontrado na rede.")

        #! Executa o fluxo de carga e imprime resultados para os trafos desligados
        if show_prints:
            self.executar_fluxo_de_carga()

            # Calcula potência aparente após alterações
            self.calcular_potencia_aparente_trafos()
            self.imprimir_resultados()

    #! Funções matematicas
    def calcular_potencia_aparente_trafos(self):
        """Calcula a potência aparente nos transformadores."""
        if not self.net.res_trafo.empty:
            if 's_aparente_hv_mva' not in self.net.res_trafo.columns:
                self.net.res_trafo['s_aparente_hv_mva'] = 0
            if 's_aparente_lv_mva' not in self.net.res_trafo.columns:
                self.net.res_trafo['s_aparente_lv_mva'] = 0

            # Calculando potência aparente para alta e baixa tensão
            potencia_high_tensao = (self.net.res_trafo['p_hv_mw']**2 + self.net.res_trafo['q_hv_mvar']**2)**0.5
            potencia_baixa_tensao = (self.net.res_trafo['p_lv_mw']**2 + self.net.res_trafo['q_lv_mvar']**2)**0.5

            return potencia_high_tensao, potencia_baixa_tensao
        else:
            print("Nenhum transformador na rede para calcular potência aparente.")
            return []

    def calcular_potencia_aparente_linhas(self):
        """Calcula a potência aparente nas linhas."""
        return (self.net.res_line['p_from_mw']**2 + self.net.res_line['q_from_mvar']**2)**0.5


    def religar_todos_os_ramos_agendamento(self):
        """Religa todos os ramos (linhas e transformadores) da rede elétrica."""
        # Religa todas as linhas
        self.net.line['in_service'] = True

        # Religa todos os transformadores
        self.net.trafo['in_service'] = True

        self.log("Todos os ramos religados.", level="success")



    def imprimir_resultados(self):
        """Retorna um array com todos os dados da rede elétrica"""
        dataframe = pd.DataFrame()

        # Calculo de potencia e colocando uma nova tabela no pandapower
        high_power_transformador, lower_power_transformador = self.calcular_potencia_aparente_trafos()
        fluxo_potencia_aparente_linhas = self.calcular_potencia_aparente_linhas()

        self.net.res_trafo['s_aparente_hv_mva'] = high_power_transformador
        self.net.res_trafo['s_aparente_lv_mva'] =  lower_power_transformador

        if self.debug:

            self.show_status()


        dataframe["tensao_nos_barramentos"] =  self.net.res_bus[['vm_pu']]
        dataframe["potencia_aparente_nas_linhas"] =  fluxo_potencia_aparente_linhas
        dataframe["porcentagem_de_carga_nas_linhas"] =  self.net.res_line[['loading_percent']]
        dataframe["potencia_aparente_nos_transformadores"] =  self.net.res_trafo[['p_hv_mw']]
        dataframe["porcentagem_de_carga_nos_transformadores"] =  self.net.res_trafo[['loading_percent']]

        dataframe.to_excel("dados_rede_eletrica.xlsx")
        self.log("\n\n\nDados da rede eletrica em formato de tabela excel disponivel!")

        return dataframe


## Main Function
Pode ser modificada para:
- Considerar toda a main como uma função objetivo, sem debug e retornando 'fitness_final'
- Criar casos para os dados dos sistemas elétricos implementados em Zanghi(2016) - IEEE30, IEEE57, IEEE118 e SIN45
- Só executar o fluxo de potência caso a entrada de 'bd_aptidao_cenario' ainda não tenha sido calculada
- Criação da 'bd_aptidao_cenario' fora do escopo da função objetivo para permitir o acesso à base em diversas execuções do algoritmo de otimização

In [ ]:
#! 1) Criar a rede elétrica IEEE 14 barras, Inicializar a classe com a rede e carrega a tabela de agendamento
rede = RedeEletricaPandaPower("14", debug=False)
#display(rede.net)

#! Colocando pesos como input do usuario e os dados de entrada do agendamento
rede.pesos["tensao"] = {"min": 100, "max": 100}
rede.pesos["loading_linhas"] = 100
rede.pesos["loading_trafos"] = 100

# Tabela agendamentos em xlsx hardcoded
agendamento_df = pd.DataFrame([
    {"ramo": [1, 4], "inicio": "14:00", "duracao": 6 ,"prioridade": 4},
    {"ramo": [1, 3], "inicio": "13:00", "duracao": 5, "prioridade": 1},
    {"ramo": [3, 6], "inicio": "12:00", "duracao": 6, "prioridade": 1},
    {"ramo": [11, 12], "inicio": "24:00", "duracao": 6, "prioridade": 1},
    {"ramo": [9, 10], "inicio": "18:00", "duracao": 4, "prioridade": 1}
])

contingencia_df = pd.DataFrame([
        {"contingencia":1,  "from":2 , "to": 3},
        {"contingencia":2,  "from":5 , "to": 12},
        {"contingencia":3,  "from":12 , "to": 13},
 ])

# Converter horários de início para horas do dia
agendamento_df['inicio'] = agendamento_df['inicio'].apply(lambda x: int(x.split(':')[0]))

# Calcular horário de término em horas do dia
agendamento_df['final'] = agendamento_df.apply(lambda row: (row['inicio'] + row['duracao']) % 24, axis=1)

# Calcular a duração total do agendamento em horas
duracao_total_agendamento = agendamento_df['final'].sum()

rede.validar_dados(agendamento_df, contingencia_df)

# 2)  Avaliar cenários e criar matriz de cenários
matriz_cenarios = rede.avalia_cenarios(
        #horas=agendamento_df['final'].sum(), # TODO -> verificar se horario fixo de 32 pode se expandir
        horas = 32,
        hora_inicio=agendamento_df['inicio'],
        duracao=agendamento_df['duracao'],
        ls=0, le=8,
        ms=8, me=18,
        hs=18, he=24
    )

#! Calculo  de otimização para achar o fitness de cada cenario
# Inicializar variáveis para cálculo de violações
violacoes_total = []
violacoes_hash_table = {}
#print("\nTabela agendamento")
#display(agendamento_df)

# Generate hash key (teste 01)
contingencias = contingencia_df['contingencia'].to_list()
num_carregamentos = 3
num_contingencias = len(contingencias) # 3
num_desligamentos = len(agendamento_df) # 5

# FAZENDO UM BANCO EM MEMORIA DE EXECUÇÃO
bd_aptidao_cenario =[-1.0]*(num_contingencias* num_carregamentos*(2**num_desligamentos) )

try:
    # 3) Processar cada cenário da matriz de cenários
    for cenario in matriz_cenarios:
        perfil = cenario[0]
        estado_ramos = cenario[1:]


        # 4) Ajustar carregamento para o perfil do cenário
        rede.ajustar_cargas(perfil)


        # Loop through contingencies before calculating violations for the scenario
        for contingencia_atual in range(num_contingencias):
            contingencia_atual += 1

            #5)  Ligar todos os ramos antes de aplicar mudanças
            rede.religar_todos_os_ramos_agendamento()

            # 6) Fazendo os deligamentos com base na tabela em .xlsx e nos cenários calculados
            rede.desligar_elementos_agendamento(estado_ramos)

            # 7) Identifica ramos afetados pela contingência
            ramo_contingencia = list(contingencia_df.loc[contingencia_df['contingencia'] == contingencia_atual, ['from', 'to']].values[0])
            rede.log(f"\n{contingencia_atual}) Ramo da contingencia = { ramo_contingencia}\n")

            # 8) Desliga os ramos afetados
            rede.desligar_contingencia(ramo_contingencia)

            # 9) Executar fluxo de potência para o cenário com contingência
            if rede.executar_fluxo_de_carga():

                # 10) Calcular violações com pesos e armazenar os resultados
                fitness, violacoes_df = rede.calcular_violacoes_fitness()
                violacoes_total.append(fitness)

            else:
                fitness = rede.pesos["demanda"] # penalidade com valor default de 99

            # 11) Store violation in the hash table
            hash_key = rede.hashtableindex(perfil, num_carregamentos, contingencia_atual, num_contingencias, estado_ramos)

            violacoes_hash_table[hash_key] = fitness

            bd_aptidao_cenario[hash_key] = fitness
            rede.log(f"Hash key = { hash_key}\n")


        #! Ver apenas o true in service de barras e transformadores
        #rede.show_status()

    #! Usando dicionario nos temos os valores acumulando tirando os valores nulos
    hash_df2 = pd.DataFrame(violacoes_hash_table.items(), columns=['Hash Key', 'Fitness'])

    # Passando os valores do array direto no dataframe com os index como chave (hash = chave, valor)
    hash_df = pd.DataFrame(bd_aptidao_cenario, columns=[ 'Fitness'])
    filtered_hash_table = hash_df.loc[hash_df['Fitness'] > 0]

    hash_df.to_excel("hash_table.xlsx", index=False)

    # 12) Calcular fitness final com somatorio das vioações com pesos de todos os cenarios
    fitness_final = sum(violacoes_total)

    rede.log(f"\nFitness do agendamento = {fitness_final:.2f}\n")


except Exception as e:
    print(f"\nErro: {e}")

Visualização de dados do último cenário do agendamento

In [ ]:
from pandapower.plotting import simple_plot, simple_plotly, pf_res_plotly

results = rede.imprimir_resultados()
display(results)


pf_res_plotly(rede.net)

,tensao_nos_barramentos,potencia_aparente_nas_linhas,porcentagem_de_carga_nas_linhas,potencia_aparente_nos_transformadores,porcentagem_de_carga_nos_transformadores
0,1.060000,223.427878,2.129101,3.871833e+01,0.395958
1,1.045000,101.901235,0.971043,2.205120e+01,0.222502
2,1.010000,97.204017,0.939578,5.469065e+01,0.554548
3,1.006567,73.456668,0.710035,2.131628e-14,0.243076
4,1.009937,53.451053,0.519028,3.871833e+01,0.398764
5,1.070000,32.844970,0.333922,NaN,NaN
6,1.047610,85.262344,0.855617,NaN,NaN
7,1.090000,15.864208,0.149761,NaN,NaN
8,1.033146,8.333433,0.078669,NaN,NaN
9,1.029508,19.530264,0.184370,NaN,NaN
